
# Exercises XP - Diabetes Classification

## What you will learn
- Understanding the problem
- Data collection
- Model training for classification
- Model evaluation

## What you will create
- A Logistic Regression model to predict diabetes



## Exercise 1 - Understanding the problem and Data Collection

We want to predict if an individual has diabetes.

- Load the diabetes dataset and explore it
- Count positive and negative cases
- Split the data into train and test


In [ ]:

# TODO: load the dataset
# If running on Colab, upload the zip or csv then adjust the path
import pandas as pd
from google.colab import files

print('Please upload diabetes_prediction_dataset.csv:')
uploaded = files.upload()

df = pd.read_csv('diabetes_prediction_dataset.csv')

print(df.shape)
display(df.head())
print(df.dtypes)
print("Missing per column:")
display(df.isna().sum().sort_values(ascending=False))


In [ ]:
# Assume target column is named 'diabetes' with 0 or 1 values
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"
print(df['diabetes'].value_counts())


In [ ]:

# TODO: train test split
from sklearn.model_selection import train_test_split
X = df.drop(columns=['diabetes'])
y = df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)



## Exercise 2 - Model picking and standardization

- Which model can we use and why
- Do we need to standardize
- If yes, apply StandardScaler



**Logistic Regression** is well-suited for this problem because the target variable `diabetes` is binary (0 = non-diabetic, 1 = diabetic), and Logistic Regression is specifically designed to model binary outcomes by estimating the probability P(diabetes=1) through a linear decision boundary passed through the sigmoid function. This produces calibrated probabilities, meaning the model output can be directly interpreted as the likelihood that a patient has diabetes — which is valuable in a clinical context. The model is also highly interpretable: each coefficient shows how a one-unit change in a feature shifts the log-odds of diabetes, making it explainable to healthcare professionals.

**Standardization is necessary** because the numeric features in this dataset have very different scales: `blood_glucose_level` ranges from roughly 80 to 300, `age` from 0 to 80, `bmi` from ~10 to ~95, and `HbA1c_level` from ~3.5 to 9. Without standardization, features with large numeric ranges dominate the optimization and cause slower, less stable convergence of the gradient-based solver. Applying `StandardScaler` on numeric columns (`age`, `hypertension`, `heart_disease`, `bmi`, `HbA1c_level`, `blood_glucose_level`) transforms each to mean = 0 and std = 1. The two categorical columns (`gender` and `smoking_history`) are handled with `OneHotEncoder` since they are string values that must be converted to numeric form.


In [ ]:
# TODO: build a preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print("Categorical:", cat_cols)
print("Numeric:", num_cols)


## Exercise 3 - Model training

In [ ]:

# TODO: train Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ('preprocessor', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
clf.fit(X_train, y_train)
print('Model trained successfully!')



## Exercise 4 - Evaluation metrics

- Plot accuracy and comment
- Plot confusion matrix and comment
- Plot precision, recall, F1 and comment


In [ ]:
# TODO: use the metrics functions properly to plot the scores
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("Accuracy:",  round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall:",    round(rec, 4))
print("F1:",        round(f1, 4))

# Simple bar plot of metrics
import matplotlib.pyplot as plt
plt.figure()
plt.bar(['accuracy','precision','recall','f1'], [acc,prec,rec,f1])
plt.title('Metrics on test')
plt.ylabel('Score')
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title('Confusion matrix')
plt.show()


The bar chart shows that **accuracy** is the highest metric, but it is misleading here because the dataset is heavily imbalanced — out of 100,000 patients, approximately 91,500 are non-diabetic and only 8,500 are diabetic (~8.5%). A naive model that always predicts non-diabetic would already reach ~91.5% accuracy without detecting a single diabetic case, so accuracy alone is not a reliable measure of performance for this dataset.

**Precision** tells us what fraction of patients the model flags as diabetic are truly diabetic — high precision means few false alarms. **Recall** tells us what fraction of actual diabetic patients the model catches — this is the most critical metric in a medical screening context because a low recall means real diabetic patients are missed and go untreated. The **F1-score** is the harmonic mean of precision and recall and is the most informative single metric when classes are imbalanced. If recall is noticeably lower than precision, the model is being too conservative in flagging positive cases, which is clinically dangerous and would warrant lowering the classification threshold or applying `class_weight='balanced'` to the Logistic Regression.


## Exercise 5 - Visualizing the performance of our model

Visualize a 2D decision boundary with accuracy info. Use only two informative features for this plot to keep it 2D. Suggested pair: `HbA1c_level` and `blood_glucose_level` if present. Otherwise pick any two numeric features.


In [ ]:
# TODO: If these columns do not exist, change `feat_x` and `feat_y` below to two numeric features that exist in your data.
import numpy as np
import matplotlib.pyplot as plt

feat_x = 'HbA1c_level' if 'HbA1c_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x, feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0,1])], remainder='drop')),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Mesh
x_min, x_max = X2_train[feat_x].min()-1, X2_train[feat_x].max()+1
y_min, y_max = X2_train[feat_y].min()-1, X2_train[feat_y].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

plt.figure(figsize=(6,5))
cs = plt.contour(xx, yy, probs, levels=[0.5])
plt.clabel(cs, inline=True, fmt={0.5:'P=0.5'})
plt.scatter(X2_test[feat_x], X2_test[feat_y], c=y_test, alpha=0.7)
plt.xlabel(feat_x); plt.ylabel(feat_y)
from sklearn.metrics import accuracy_score
acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))
plt.title(f'Decision boundary on 2 features - test accuracy {acc2:.3f}')
plt.show()



## Exercise 6 - ROC curve

Use the code template provided to plot the ROC curve for your model and compute AUC. You can reuse the fitted `clf` pipeline.

Template summary:
- Get predicted probabilities for the positive class
- Compute fpr and tpr with `roc_curve`
- Plot ROC and print AUC


In [ ]:

from sklearn import metrics
y_proba = clf.predict_proba(X_test)[:,1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure()
plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc=4)
plt.title('ROC curve')
plt.show()


The **ROC curve** plots the True Positive Rate (Recall — the fraction of actual diabetic patients correctly identified) against the False Positive Rate (the fraction of healthy patients incorrectly flagged as diabetic) at every possible classification threshold. A model with no predictive power produces a diagonal line from (0,0) to (1,1), corresponding to AUC = 0.5 — no better than random guessing. Our model's curve bows toward the upper-left corner, confirming that features like `HbA1c_level` and `blood_glucose_level` carry strong predictive signal for diabetes.

The **AUC (Area Under the Curve)** summarizes the model's overall discriminative ability in a single number: 1.0 = perfect, 0.9+ = excellent, 0.8+ = good, 0.7+ = fair, 0.5 = random. Because the dataset is heavily imbalanced (~91.5% non-diabetic), a high AUC is especially meaningful — it confirms the model ranks true diabetic patients above healthy ones across all thresholds, not just at the default 0.5 cutoff. In clinical practice, a physician would use this curve to select a lower threshold (e.g., 0.3) to maximize recall and catch as many diabetic patients as possible, accepting a higher false positive rate in exchange for fewer missed diagnoses.